In [81]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt

pcd_path = "Phase_2_PCD/PointCloud_e5.pcd" # <- full path to your .pcd file
intrinsics = dict(fx=617.0, fy=617.0, cx=319.5, cy=239.5) # <- camera dimensions

# bounding-box supplied by the CV model (x1, y1, x2, y2), float is fine
# Box_corners = [338.89346, 268.44052, 519.0965 , 439.19873] # e2
Box_corners = [182.18625, 223.96574, 460.8647 , 412.31375] # box coords
bbox_pix = np.array(Box_corners) 

# load the point cloud
pcd = o3d.io.read_point_cloud(pcd_path)
pts = np.asarray(pcd.points) # shape = (N,3)

In [82]:
# Visualize raw point cloud
o3d.visualization.draw_geometries([pcd], window_name="Raw Point Cloud")

### Find points that fall in 2-D Bounding Box from CV script

In [83]:
eps = 1e-9
u = (intrinsics["fx"] * pts[:,0]) / (pts[:,2] + eps) + intrinsics["cx"]
v = (intrinsics["fy"] * pts[:,1]) / (pts[:,2] + eps) + intrinsics["cy"]
uv = np.c_[u, v]      

x1, y1, x2, y2 = bbox_pix
inside = (uv[:,0] >= x1) & (uv[:,0] <= x2) & \
         (uv[:,1] >= y1) & (uv[:,1] <= y2)

print(f"Total points in cloud : {len(pcd.points):,}")
print(f"Points that fall in 2-D BBox : {inside.sum():,}")

col = np.zeros((len(pcd.points),3))
col[:] = [0.7,0.7,0.7]          # grey
col[inside] = [1, 0.706, 0]   # Yellow
pcd.colors = o3d.utility.Vector3dVector(col)

o3d.visualization.draw_geometries([pcd], window_name="Points that fall in 2-D BBox")

Total points in cloud : 307,200
Points that fall in 2-D BBox : 57,124


### Filters, fix tilt, and segment object / table

In [84]:
# Convert units (assumes points are in mm)
MM_TO_M = 1 / 1000.0

points = np.asarray(pcd.points, dtype=np.float64) * MM_TO_M
pcd.points = o3d.utility.Vector3dVector(points)

In [85]:
# variant which filters and then fix tilt
def level_and_filter(pcd: o3d.geometry.PointCloud,
                     distance_thresh=0.002,
                     num_iters=1000,
                     z_min=0.005,
                     z_max=0.025) -> o3d.geometry.PointCloud:
    """
    1) RANSAC‐fit the dominant plane (table) and rotate it horizontal.
    2) Remove all points whose Z is below (table_z + z_margin).
    """
    # Fit table and build leveling transform
    (a, b, c, d), inliers = pcd.segment_plane(distance_thresh,
                                              ransac_n=3,
                                              num_iterations=num_iters)
    # pick one table inlier as pivot
    centroid = np.asarray(pcd.points)[inliers[0]]
    # normal → unit
    n = np.array([a, b, c])
    n /= np.linalg.norm(n)
    # build Rodrigues rotation to send n → [0,0,1]
    z = np.array([0, 0, 1.0])
    v = np.cross(n, z)
    s = np.linalg.norm(v)
    if s < 1e-6:
        R = np.eye(3)
    else:
        c_dot = np.dot(n, z)
        vx = np.array([[   0, -v[2],  v[1]],
                       [ v[2],    0, -v[0]],
                       [-v[1],  v[0],   0]])
        R = np.eye(3) + vx + vx @ vx * ((1 - c_dot) / (s**2))

    # apply centering → rotation → decentering
    pcd = pcd.translate(-centroid, relative=False)
    pcd = pcd.rotate(R, center=(0, 0, 0))
    pcd = pcd.translate(centroid, relative=False)

    # Height filter in leveled frame
    # after leveling, the table plane goes through z = centroid[2]
    table_z = centroid[2]
    print(table_z)
    pts = np.asarray(pcd.points)

    # Initialize mask to keep all points first
    mask = np.ones(len(pts), dtype=bool)

    if z_min is not None:
        mask &= pts[:, 2] <= (table_z + z_min) # filter out all points below the segmented table + certain height
    if z_max is not None:
        mask &= pts[:, 2] >= (table_z - z_max) # filter out all points above the segmented table + certain height

    return pcd.select_by_index(np.where(mask)[0])

pcd = level_and_filter(pcd,
                        distance_thresh=0.003, # how tightly the plane fitting should hug the data, 2mm
                        num_iters=1000,
                        z_min=0, # 25mm (0.025) default, extra margin at bottom of table for filtering below
                        z_max=0.250) # 250mm (0.250) default, filter all points above the table

o3d.visualization.draw_geometries([pcd],
                                  window_name="Leveled + Z‐filtered")

-0.5718550735706946


In [86]:
# Fit a plane to the point cloud using RANSAC, will get table height
def plane_fit_horizontal(cloud, thresh=0.005, n_iter=1000, max_tilt_deg=15):
    (a, b, c, d), inliers = cloud.segment_plane(thresh, ransac_n=3, num_iterations=n_iter)
    
    # Normalize
    n = np.array([a, b, c])
    norm = np.linalg.norm(n)
    n /= norm
    d /= norm

    # Check angle between plane normal and Z-axis
    cos_theta = np.abs(np.dot(n, [0, 0, 1]))  # |cos(θ)| where θ is angle with Z
    angle_deg = np.arccos(cos_theta) * (180.0 / np.pi)

    if angle_deg > max_tilt_deg:
        print(f"Rejected plane: tilt angle {angle_deg:.2f}° > {max_tilt_deg}° (not horizontal enough)")
        return None, None, None

    return n, d, inliers

n_tab, d_tab, table_inliers = plane_fit_horizontal(pcd, thresh=0.005) # Finds the table plane, extract normal vecotr (n_tab) and offset (d_tab), thresh -> change plane volume (will effects final height)

# check if there is table in the point cloud (horizontal plane only)
if len(table_inliers) == None:
    print("No table detected")
    exit()

table_pc = pcd.select_by_index(table_inliers) # table object (dominant flat plane)
obj_pc = pcd.select_by_index(table_inliers, invert=True) # all other object, found by using invert=True

# Visualize
table_pc.paint_uniform_color([1, 0, 0]) # red for table
o3d.visualization.draw_geometries([table_pc, obj_pc], window_name="Table (Red) and Object (Gray)")

In [87]:
o3d.visualization.draw_geometries([obj_pc]) # visualize just one item

### Clustering isolation base on CV dimensions

In [88]:
# Old method, choose largest in general
labels = np.array(obj_pc.cluster_dbscan(eps=0.01,
                                        min_points=20,
                                        print_progress=True))

largest = np.bincount(labels[labels >= 0]).argmax() # Finds the most populated cluster (assumed to be the box)
box_pc = obj_pc.select_by_index(np.where(labels == largest)[0]) # Selects only the points that belong to the largest cluster.

# Visualize
box_pc.paint_uniform_color([0.2, 0.8, 1.0])
o3d.visualization.draw_geometries([box_pc], window_name="Detected Box Cluster")

In [89]:
# New method, choose largest in yellow area
# DBSCAN on the object cloud
labels = np.array(
    obj_pc.cluster_dbscan(eps=0.01, min_points=20, print_progress=True)
)

# detect yellow from colours in obj_pc
col_obj = np.asarray(obj_pc.colors)
yellow_rgb = np.array([1.0, 0.706, 0.0])
yellow_obj_mask = np.all(
     np.isclose(col_obj, yellow_rgb, atol=1e-3), axis=1
)

# Find clusters with ≥50 % yellow points and keep the largest one
THRESHOLD = 0.50 # 50 %
best_lbl   = -1
best_size  = -1

for lbl in np.unique(labels[labels >= 0]):     # iterate over real clusters
    idx   = (labels == lbl)                    # points in this cluster
    ratio = yellow_obj_mask[idx].sum() / idx.sum()

    if ratio >= THRESHOLD and idx.sum() > best_size:
        best_lbl  = lbl
        best_size = idx.sum()

if best_lbl == -1:
    raise RuntimeError(
        f"No cluster had ≥{int(THRESHOLD*100)} % of its points in the yellow ROI."
    )

box_pc = obj_pc.select_by_index(np.where(labels == best_lbl)[0])

# Visualise
box_pc.paint_uniform_color([0.2, 0.8, 1.0]) # cyan
o3d.visualization.draw_geometries([box_pc],
                                  window_name="Largest cluster ≥60 % yellow")


In [90]:
o3d.visualization.draw_geometries([box_pc, obj_pc],
                                  window_name="Full")

### Fit lid + RANSAC failover for hollow

In [91]:
def plane_fit(cloud, thresh=0.005, n_iter=1000):
    (a, b, c, d), inliers = cloud.segment_plane(thresh, ransac_n=3, num_iterations=n_iter)
    
    # Normalize
    n = np.array([a, b, c])
    norm = np.linalg.norm(n)
    n /= norm
    d /= norm

    return n, d, inliers

In [92]:
def plane_fit_horizontal(cloud, thresh=0.005, n_iter=1000, max_tilt_deg=15):
    (a, b, c, d), inliers = cloud.segment_plane(thresh, ransac_n=3, num_iterations=n_iter)
    
    # Normalize
    n = np.array([a, b, c])
    norm = np.linalg.norm(n)
    n /= norm
    d /= norm

    # Check angle between plane normal and Z-axis
    cos_theta = np.abs(np.dot(n, [0, 0, 1]))  # |cos(θ)| where θ is angle with Z
    angle_deg = np.arccos(cos_theta) * (180.0 / np.pi)

    if angle_deg > max_tilt_deg:
        print(f"Rejected plane: tilt angle {angle_deg:.2f}° > {max_tilt_deg}° (not horizontal enough)")
        return None, None, None

    return n, d, inliers

In [93]:
# RANSAC to find dominant plane, this will be top of box (or side, or hollow)
n_lid, d_lid, lid_inliers = plane_fit(box_pc)
lid_pc = box_pc.select_by_index(lid_inliers)

lid_pc.paint_uniform_color([0.0, 1.0, 0.0]) # Green
box_pc.paint_uniform_color([0.5, 0.5, 0.5]) # Gray
o3d.visualization.draw_geometries([lid_pc, box_pc], window_name="Lid (Green) on Box")

In [94]:
from scipy.spatial import ConvexHull

def compute_2d_obb_from_lid(lid_pc: o3d.geometry.PointCloud):
    # Get XY coordinates of lid points (project to 2D)
    points = np.asarray(lid_pc.points)
    xy = points[:, :2]  # Ignore Z

    # Compute convex hull
    hull = ConvexHull(xy)
    hull_pts = xy[hull.vertices]

    # Rotating calipers: test all edges of the hull
    min_area = float("inf")
    best_rect = None

    for i in range(len(hull_pts)):
        # Edge vector
        edge = hull_pts[(i + 1) % len(hull_pts)] - hull_pts[i]
        edge /= np.linalg.norm(edge)  # normalize

        # Get orthogonal vector (rotate 90°)
        ortho = np.array([-edge[1], edge[0]])

        # Build rotation matrix to align edge with X-axis
        R = np.stack([edge, ortho]).T

        # Rotate all hull points
        rot_pts = hull_pts @ R

        # Get bounding box in this frame
        min_xy = rot_pts.min(axis=0)
        max_xy = rot_pts.max(axis=0)
        extent = max_xy - min_xy
        area = extent[0] * extent[1]

        # Update if this is the smallest area
        if area < min_area:
            min_area = area
            best_rect = (R, min_xy, max_xy)

    # Recover the best rectangle in world coords
    R, min_xy, max_xy = best_rect
    center_2d = (min_xy + max_xy) / 2
    corners_2d = np.array([
        [min_xy[0], min_xy[1]],
        [max_xy[0], min_xy[1]],
        [max_xy[0], max_xy[1]],
        [min_xy[0], max_xy[1]],
    ])
    world_corners = (corners_2d @ R.T)

    # Estimate average Z height of lid for 3D placement
    z_mean = np.mean(points[:, 2])
    corners_3d = np.column_stack([world_corners, np.full(4, z_mean)])

    # Return rectangle corners and length/width
    length, width = np.abs(max_xy - min_xy)
    return corners_3d, length, width

In [95]:
from scipy.spatial import ConvexHull
def top_percentile_obb(box_pc, percentile=10):
    """
    1. Keep only the top `percentile`% of points by Z.
    2. Wrap them into a temporary PointCloud.
    3. Call compute_2d_obb_from_lid on that cloud to get corners, length, width.
    """
    # Pull out raw points
    pts = np.asarray(box_pc.points)
    zs  = pts[:, 2]

    # Threshold at the desired percentile
    z_thr   = np.percentile(zs, percentile)
    top_idx = np.where(zs <= z_thr)[0]
    top_pts = pts[top_idx]               # shape (M, 3)

    # Build a temp Open3D cloud of just those top points
    temp_pc = o3d.geometry.PointCloud()
    temp_pc.points = o3d.utility.Vector3dVector(top_pts)

    # Now call your existing plane‐to‐OBB function
    corners, length, width = compute_2d_obb_from_lid(temp_pc)

    return corners, length, width

In [96]:
# RANSAC to find dominant plane (lid) + hollow check
n_lid, d_lid, lid_inliers = plane_fit_horizontal(box_pc)
lid_pc = box_pc.select_by_index(lid_inliers)

# Project inliers to XY and compute their 2D convex hull
from scipy.spatial import ConvexHull
from matplotlib.path import Path

pts2d = np.asarray(lid_pc.points)[:, :2]
hull = ConvexHull(pts2d)
hull_pts = pts2d[hull.vertices]

# Build a 2D polygon for point-in-hull tests
poly = Path(hull_pts)

# Create a grid covering the hull’s bounding box
grid_size = 50
min_xy = hull_pts.min(axis=0)
max_xy = hull_pts.max(axis=0)
cell_size = (max_xy - min_xy) / grid_size

# Which grid‐cell centers are inside the hull?
x_centers = min_xy[0] + (np.arange(grid_size) + 0.5) * cell_size[0]
y_centers = min_xy[1] + (np.arange(grid_size) + 0.5) * cell_size[1]
XX, YY = np.meshgrid(x_centers, y_centers)
centers = np.vstack([XX.ravel(), YY.ravel()]).T
inside = poly.contains_points(centers).reshape(grid_size, grid_size)

# Mark which cells have at least one lid point
filled = np.zeros_like(inside)
idx = ((pts2d - min_xy) / cell_size).astype(int)
idx[:, 0] = np.clip(idx[:, 0], 0, grid_size - 1)
idx[:, 1] = np.clip(idx[:, 1], 0, grid_size - 1)
# Note: idx rows are [i, j] = [x_index, y_index]
filled[idx[:, 1], idx[:, 0]] = True

# Compute coverage = filled cells inside hull / total hull cells
coverage = filled[inside].sum() / inside.sum()
print(f"Lid coverage: {coverage:.2%}")

Lid coverage: 84.26%


In [97]:
# If RANSAC hull is too hollow or no inliers (if angle is too tilted for RANSAC), fall back to top‐percentile outline
if (coverage < 0.7) or (lid_inliers == None):
    corners, length, width = top_percentile_obb(box_pc, percentile=20) # top 20 percent of box is used to project to 2D then find XY
    # Compute height from the top 5% of points
    pts = np.asarray(box_pc.points)
    zs  = pts[:, 2]

    # want the top 1%, so use 99th percentile threshold, for height level
    z_thr   = np.percentile(zs, 1)
    top_idx = np.where(zs <= z_thr)[0]
    top_pts = pts[top_idx]

    # Use the mean (or max) Z of those as your lid height
    z_lid_est = top_pts[:, 2].mean()

    # Get diff between top of box and table
    height_m = abs(z_lid_est + d_tab)

else:
    corners, length, width = compute_2d_obb_from_lid(lid_pc) # standard method of projecting to 2D then find XY

    # Get diff between top of box and table
    height_m = abs(d_lid - d_tab)

print(f"Box ≈ {length*100:.2f} × {width*100:.2f} × {height_m*100:.2f} cm  (L×W×H)")

Box ≈ 24.68 × 17.30 × 7.13 cm  (L×W×H)


In [98]:
# Percentage threshold visualization test -> dont forget the PCD is upside down so logic is wonky
# Assume box_pc is your Open3D PointCloud of the box cluster
pts = np.asarray(box_pc.points)
zs  = pts[:, 2]

# 1) Compute 80th‐percentile threshold
z_thr   = np.percentile(zs, 20)

# 2) Split the cloud into 'top' and 'rest'
top_idx  = np.where(zs < z_thr)[0]
rest_idx = np.where(zs >=  z_thr)[0]

top_pc  = box_pc.select_by_index(top_idx)
rest_pc = box_pc.select_by_index(rest_idx)

# 3) Color them differently
top_pc.paint_uniform_color([1.0, 0.0, 0.0])    # red for top 10%
rest_pc.paint_uniform_color([0.7, 0.7, 0.7])   # light gray for the rest

# 4) Visualize together
o3d.visualization.draw_geometries(
    [rest_pc, top_pc],
    window_name="Box Cluster: Top 10% Highlighted",
    width=800, height=600
)

In [99]:
# Optional: create and visualize a line set for the box
lines = [[0,1],[1,2],[2,3],[3,0]]
colors = [[1, 0.5, 0] for _ in lines]  # orange

line_set = o3d.geometry.LineSet(
    points=o3d.utility.Vector3dVector(corners),
    lines=o3d.utility.Vector2iVector(lines),
)
line_set.colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_geometries([top_pc, line_set]) # lid_pc for non-hollow, top_pc for hollow

### Final visualization

In [100]:
# Create table bounding boxes
table_obj = table_pc.get_oriented_bounding_box()

# Set colors
table_obj.color = (1.0, 0.0, 0.0)
table_pc.paint_uniform_color([1.0, 0.0, 0.0]) # table = red
lid_pc.paint_uniform_color([0.0, 1.0, 0.0]) # lid = green
box_pc.paint_uniform_color([0.2, 0.8, 1.0]) # box body = cyan

o3d.visualization.draw_geometries(
    [table_pc, lid_pc, box_pc, line_set, table_obj, obj_pc],
    window_name="Final Visualization: Table, Lid, Box, OBB",
    width=800, height=600
)

[Open3D WARNING] [ViewControl] SetViewPoint() failed because window height and width are not set.
